# ElegyBox — Note Model Training (cross-bar conditioning)

Trains the **note-filling model** with cross-bar memory.

### What changed in this version
- **Cross-bar prefix**: the prefix is now `[prev_root, prev_qual, <prev_bar_note_tokens>, curr_root, curr_qual]` instead of just 4 chord tokens. The model sees what was actually played in bar N-1 when generating bar N, giving it genuine melodic memory across bar lines.
- **Larger context window**: `seq_len` increased from 256 → 512 to accommodate the extended prefix without dropping dense bars.
- **Updated augmentation**: `_transpose` now shifts pitch tokens inside the prefix (the prev-bar note context) in addition to chord roots and current-bar pitches, keeping the transposition consistent.
- The **chord model is unchanged** — only `note_model.pt` needs to be replaced after this run.

### Before you run anything
1. On your **local machine**, pull the latest code and re-run preprocessing to rebuild training data with the new prefix format:
   ```
   python preprocess.py
   ```
   Then compress for upload:
   ```python
   import gzip, pickle, shutil
   with open('data/processed/note_samples.pkl','rb') as f_in, \
        gzip.open('data/processed/note_samples.pkl.gz','wb',compresslevel=6) as f_out:
       shutil.copyfileobj(f_in, f_out)
   ```
2. In the **left panel file browser**, drag-and-drop **`note_samples.pkl.gz`** into this studio.
3. Run cells **top to bottom**.
4. **Cell 9** gives you a download link for `note_model.pt` → place it in your local `ElegyBox/checkpoints/`.

---

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'tqdm'], check=True)
print('✓ Dependencies ready')

## Cell 2 — Verify GPU

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✓ GPU: {name}  ({mem:.1f} GB VRAM)')
else:
    print('⚠ No GPU found — training will be slow.')

## Cell 3 — Load training data

Make sure **`note_samples.pkl.gz`** is uploaded (visible in the left file browser).

The sanity check prints average prefix length — it should be **~30+ tokens** (cross-bar context included). A value of 4 means the old tokenizer was used; go back and re-run `preprocess.py` locally.

In [ ]:
import pickle, gzip, os

DATA_PATH = 'note_samples.pkl.gz'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"'{DATA_PATH}' not found.\n"
        "→ Drag-and-drop note_samples.pkl.gz from your local data/processed/ folder "
        "into the file browser on the left, then re-run this cell."
    )

with gzip.open(DATA_PATH, 'rb') as f:
    note_samples = pickle.load(f)

print(f'✓ Loaded {len(note_samples):,} bar samples')

# Sanity check: confirm new prefix format
sample_prefixes = [len(p) for p, _ in note_samples[:500]]
avg_prefix = sum(sample_prefixes) / len(sample_prefixes)
print(f'  Avg prefix length (first 500 samples): {avg_prefix:.1f} tokens')
if avg_prefix < 10:
    print('  ⚠ WARNING: prefix looks like old 4-token format.')
    print('    Re-run preprocess.py locally with the updated tokenizer.py, then re-upload.')
else:
    print('  ✓ Extended prefix format confirmed (cross-bar context present)')
print(f'  Example — prefix: {len(note_samples[0][0])} tok,  notes: {len(note_samples[0][1])} tok')

## Cell 4 — Model, dataset, and training code

All code is self-contained here — nothing to edit.

In [ ]:
import math, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.notebook import tqdm

# ── Vocabulary constants (must match tokenizer.py) ──────────────────
NOTE_PAD     = 0
NOTE_BAR_END = 1
NOTE_VOCAB   = 153

_NOTE_ROOT_OFF = 2
_NOTE_ON_OFF   = 41
_NOTE_DUR_OFF  = 129
_N_ROOTS       = 12
_N_PITCHES     = 88


# ── Dataset ─────────────────────────────────────────────────────────
class NoteDataset(Dataset):
    def __init__(self, samples, seq_len=512, augment=True, max_shift=5):
        self.seq_len   = seq_len
        self.augment   = augment
        self.max_shift = max_shift
        self.samples   = []
        for prefix, notes in samples:
            if len(prefix) + len(notes) <= seq_len + 1:
                self.samples.append((list(prefix), list(notes)))

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _transpose(prefix, notes, shift):
        new_prefix = list(prefix)
        for i, tok in enumerate(new_prefix):
            if _NOTE_ROOT_OFF <= tok < _NOTE_ROOT_OFF + _N_ROOTS:
                new_prefix[i] = _NOTE_ROOT_OFF + (tok - _NOTE_ROOT_OFF + shift) % _N_ROOTS
            elif _NOTE_ON_OFF <= tok < _NOTE_DUR_OFF:
                new_pitch = max(0, min(_N_PITCHES - 1, (tok - _NOTE_ON_OFF) + shift))
                new_prefix[i] = _NOTE_ON_OFF + new_pitch
        new_notes = list(notes)
        for i, tok in enumerate(new_notes):
            if _NOTE_ON_OFF <= tok < _NOTE_DUR_OFF:
                new_pitch = max(0, min(_N_PITCHES - 1, (tok - _NOTE_ON_OFF) + shift))
                new_notes[i] = _NOTE_ON_OFF + new_pitch
        return new_prefix, new_notes

    def __getitem__(self, idx):
        prefix, notes = self.samples[idx]
        if self.augment and self.max_shift > 0:
            shift = random.randint(-self.max_shift, self.max_shift)
            if shift != 0:
                prefix, notes = self._transpose(prefix, notes, shift)
        full = prefix + notes
        need = self.seq_len + 1
        if len(full) < need:
            full = full + [0] * (need - len(full))
        full = full[:need]
        x    = torch.tensor(full[:-1], dtype=torch.long)
        y    = torch.tensor(full[1:],  dtype=torch.long)
        mask = torch.zeros(len(x), dtype=torch.float)
        mask[len(prefix) - 1:] = 1.0
        mask[y == 0] = 0.0
        return x, y, mask


# ── Model ────────────────────────────────────────────────────────────
class _Block(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout,
                                          batch_first=True)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, causal_mask):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h, attn_mask=causal_mask, need_weights=False)
        x = x + h
        x = x + self.ff(self.ln2(x))
        return x


class MusicGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, seq_len, dropout=0.1):
        super().__init__()
        self.seq_len  = seq_len
        self.tok_emb  = nn.Embedding(vocab_size, d_model)
        self.pos_emb  = nn.Embedding(seq_len, d_model)
        self.drop     = nn.Dropout(dropout)
        self.blocks   = nn.ModuleList([_Block(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.ln_f     = nn.LayerNorm(d_model)
        self.head     = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        # persistent=False: recomputed from seq_len in __init__, never saved to checkpoints
        causal = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
        self.register_buffer('causal_mask', causal, persistent=False)
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, x, targets=None, loss_mask=None):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        h    = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        mask = self.causal_mask[:T, :T]
        for block in self.blocks:
            h = block(h, mask)
        logits = self.head(self.ln_f(h))
        if targets is None:
            return logits
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                               targets.reshape(-1), ignore_index=0, reduction='none')
        if loss_mask is not None:
            denom = loss_mask.reshape(-1).sum().clamp(min=1)
            loss  = (loss * loss_mask.reshape(-1)).sum() / denom
        else:
            loss = loss[targets.reshape(-1) != 0].mean()
        return logits, loss


# ── LR schedule ──────────────────────────────────────────────────────
def cosine_schedule(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.05, 0.5 * (1 + math.cos(math.pi * t)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ── Training helpers ─────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, scaler, scheduler, device):
    model.train()
    total, n = 0.0, 0
    for x, y, mask in loader:
        x, y, mask = x.to(device, non_blocking=True), \
                     y.to(device, non_blocking=True), \
                     mask.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            _, loss = model(x, y, loss_mask=mask)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item(); n += 1
    return total / max(n, 1)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total, n = 0.0, 0
    for x, y, mask in loader:
        x, y, mask = x.to(device, non_blocking=True), \
                     y.to(device, non_blocking=True), \
                     mask.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            _, loss = model(x, y, loss_mask=mask)
        total += loss.item(); n += 1
    return total / max(n, 1)


print('✓ All code loaded')

## Cell 5 — Training configuration

**Edit these values if you want to adjust training.** The defaults are tuned for a T4/A10 GPU.

`SEQ_LEN = 512` is required for the extended prefix — do not lower it.

In [ ]:
# ── Hyperparameters ── edit freely ───────────────────────────────────

EPOCHS       = 100    # stop early if val loss plateaus for 10+ epochs
BATCH_SIZE   = 128    # safe ceiling for T4 (16 GB) at seq_len=320
LR           = 4e-4   # sqrt-scaled from 3e-4 @ batch=64
WEIGHT_DECAY = 0.1
VAL_SPLIT    = 0.05
MAX_SHIFT    = 5      # pitch transposition range for augmentation (±semitones)

# ── Model size ───────────────────────────────────────────────────────
# seq_len=320: prefix is at most 52 tokens (4 chord + 48 prev-bar notes),
# leaving 268 tokens for note generation — enough for all but the densest bars.
# Attention cost scales as T², so 320 vs 512 is 2.5x less work per batch.
# IMPORTANT: models.py on your local machine must also use seq_len=320 so that
# the downloaded checkpoint loads correctly.
SEQ_LEN   = 320
MODEL_CFG = dict(
    vocab_size = NOTE_VOCAB,   # 153
    d_model    = 384,
    n_heads    = 8,
    n_layers   = 8,
    seq_len    = SEQ_LEN,
    dropout    = 0.1,
)

CHECKPOINT_PATH = 'note_model.pt'

print(f'Config: {EPOCHS} epochs, batch {BATCH_SIZE}, lr {LR}, seq_len {SEQ_LEN}')
print(f'Augmentation: pitch shift ±{MAX_SHIFT} semitones')

## Cell 6 — Build datasets and model

In [ ]:
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

n_val_raw   = max(1, int(len(note_samples) * VAL_SPLIT))
n_train_raw = len(note_samples) - n_val_raw
rng = torch.Generator().manual_seed(42)
train_idx, val_idx = random_split(range(len(note_samples)), [n_train_raw, n_val_raw],
                                  generator=rng)
train_samples = [note_samples[i] for i in train_idx]
val_samples   = [note_samples[i] for i in val_idx]

train_ds = NoteDataset(train_samples, seq_len=SEQ_LEN, augment=True,  max_shift=MAX_SHIFT)
val_ds   = NoteDataset(val_samples,   seq_len=SEQ_LEN, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} samples  |  Val: {len(val_ds):,} samples')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

model    = MusicGPT(**MODEL_CFG).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel: {n_params:,} parameters  (~{n_params/1e6:.0f} M)  seq_len={SEQ_LEN}')

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = EPOCHS * len(train_loader)
warmup_steps = min(1000, total_steps // 10)
scheduler    = cosine_schedule(optimizer, warmup_steps, total_steps)
scaler       = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

print(f'Total steps: {total_steps:,}  |  Warmup: {warmup_steps}')
print('\n✓ Ready to train')

## Cell 7 — Train!

Watch the **val loss** — the best checkpoint saves automatically when it improves.
Stop early (Kernel → Interrupt) if val loss has clearly plateaued for 10+ epochs.

In [ ]:
best_val   = float('inf')
best_epoch = 0
history    = []

epoch_bar = tqdm(range(1, EPOCHS + 1), desc='Epochs')

for epoch in epoch_bar:
    tr  = run_epoch(model, train_loader, optimizer, scaler, scheduler, device)
    val = eval_epoch(model, val_loader, device)
    history.append((tr, val))

    marker = ''
    if val < best_val:
        best_val, best_epoch = val, epoch
        torch.save({'config': MODEL_CFG, 'state': model.state_dict()}, CHECKPOINT_PATH)
        marker = '  ← saved'

    lr_now = scheduler.get_last_lr()[0]
    epoch_bar.set_postfix(train=f'{tr:.4f}', val=f'{val:.4f}', lr=f'{lr_now:.2e}')
    print(f'  [{epoch:3d}/{EPOCHS}] train={tr:.4f}  val={val:.4f}  lr={lr_now:.2e}{marker}')

print(f'\n✓ Training complete.  Best val={best_val:.4f} at epoch {best_epoch}')
print(f'  Checkpoint saved to: {CHECKPOINT_PATH}')

## Cell 8 — Plot training curve (optional)

In [ ]:
try:
    import matplotlib.pyplot as plt
    tr_hist  = [h[0] for h in history]
    val_hist = [h[1] for h in history]
    plt.figure(figsize=(9, 4))
    plt.plot(tr_hist,  label='train loss')
    plt.plot(val_hist, label='val loss')
    plt.axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5,
                label=f'best (epoch {best_epoch})')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title('Note Model Training — cross-bar conditioning')
    plt.legend(); plt.tight_layout()
    plt.savefig('training_curve.png', dpi=120)
    plt.show()
    print('Saved training_curve.png')
except ImportError:
    print('matplotlib not installed — skipping plot')

## Cell 9 — Download the checkpoint

Click the link below to download `note_model.pt`, then drop it into your local `ElegyBox/checkpoints/` folder.

In [ ]:
import os
from IPython.display import FileLink, display

size_mb = os.path.getsize(CHECKPOINT_PATH) / 1024**2
print(f'note_model.pt  ({size_mb:.1f} MB)')
print('Click the link below to download it:\n')
display(FileLink(CHECKPOINT_PATH))